# Video 03 - Reading the reply: choices, finish_reason, tokens and temperature

**You are here:** Video 3 of 13 (see `../you-are-here.svg`)

**The one idea:** the reply is more than text. **`finish_reason` tells you whether to trust it**, and `usage` tells you what it cost.

**By the end you can:**
1. Explain every part of a response.
2. Tell why the model stopped and what to do about it.
3. Read token counts and use `max_tokens` as a length and cost cap.
4. Use `temperature` to make answers predictable or creative.

**Before you start:** attach this notebook to **Serverless**.

In [0]:
from databricks.sdk import WorkspaceClient

client = WorkspaceClient().serving_endpoints.get_open_ai_client()
MODEL = "databricks-meta-llama-3-3-70b-instruct"
print("Connected. Using model:", MODEL)

## Part 1 - Anatomy of a response
Let's print **everything** the model sends back, not just the text (diagram: `../diagrams/01-anatomy-of-a-response.svg`).

In [0]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello in 5 words."}],
)

# The full response as JSON (empty fields hidden)
print(response.model_dump_json(indent=2, exclude_none=True))

In [0]:
choice = response.choices[0]          # choices is a LIST - we take the first answer

print("Text           :", choice.message.content)
print("Role           :", choice.message.role)        # always 'assistant'
print("Tool calls     :", choice.message.tool_calls)  # None for now (Video 06)
print("Finish reason  :", choice.finish_reason)       # WHY it stopped
print("Tokens in/out  :", response.usage.prompt_tokens, "/", response.usage.completion_tokens)
print("Model          :", response.model)

## Part 2 - Why `choices` is a list
With `n=2` you can ask for two answers in one call: `choices[0]` and `choices[1]`. Not every model supports this.

**Predict first:** will both models accept `n=2`?

In [0]:
for model in [MODEL, "databricks-claude-haiku-4-5"]:
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Name a fruit. One word."}],
            n=2,
            temperature=1.0,
        )
        print(f"{model}: {len(response.choices)} choices ->", [c.message.content for c in response.choices])
    except Exception as e:
        print(f"{model}: not supported -> {str(e)[:110]}")

## Part 3 - `finish_reason`: why did it stop?

| Value | Meaning | Your code should |
| --- | --- | --- |
| `stop` | Finished naturally | Use the answer |
| `length` | Hit `max_tokens` - **cut off** | Warn, raise `max_tokens`, or ask for shorter |
| `tool_calls` | Wants a tool run | Run it and send the result back (Video 06) |

**Predict first:** what happens to a 10-item list with `max_tokens=40`?

In [0]:
request = [{"role": "user", "content": "List 10 tips for learning Python, one per line."}]

for limit in [None, 40]:
    response = client.chat.completions.create(model=MODEL, messages=request, max_tokens=limit)
    choice = response.choices[0]
    print(f"=== max_tokens={limit} | finish_reason={choice.finish_reason!r} | output tokens={response.usage.completion_tokens} ===")
    print(choice.message.content, "\n")

## Part 4 - Tokens
Models read and write **tokens** - chunks of text. Rule of thumb: **1 token is about 3/4 of a word**. Tokens are what you pay for and what `max_tokens` counts.

In [0]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Write a 100-word paragraph about rivers."}],
)
text = response.choices[0].message.content
words = len(text.split())
tokens = response.usage.completion_tokens

print(f"Words written : {words}")
print(f"Output tokens : {tokens}")
print(f"Words / token : {words / tokens:.2f}   (rule of thumb: about 0.75)")

## Part 5 - Temperature: the creativity dial
Low temperature = predictable. High temperature = varied (diagram: `../diagrams/03-temperature-dial.svg`).

**Predict first:** asking the same question 5 times, how many *different* answers at temperature 0? At 1.5?

In [0]:
question = [{"role": "user", "content": "Give me one unusual hobby. Reply with 1-3 words only."}]

for temperature in [0, 1.5]:
    answers = [
        client.chat.completions.create(model=MODEL, messages=question, temperature=temperature, max_tokens=20)
        .choices[0].message.content.strip()
        for _ in range(5)
    ]
    print(f"temperature={temperature}: {len(set(answers))} different answers out of 5")
    print("   ", answers)

# Use LOW temperature for facts, extraction and agents; HIGH for brainstorming.
# Note: 0 is MOSTLY repeatable, not perfectly - never assume an LLM is deterministic.

## Cost meter: `max_tokens` is also a cost cap
Output tokens usually cost more than input tokens. `max_tokens` puts a hard ceiling on output - and therefore on cost. Prices below are **example numbers**; use your real rate.

In [0]:
PRICE_PER_1M_INPUT = 0.50    # EXAMPLE price in USD - replace with your real rate
PRICE_PER_1M_OUTPUT = 1.50   # EXAMPLE price in USD - output usually costs more

def cost(usage):
    return usage.prompt_tokens / 1e6 * PRICE_PER_1M_INPUT + usage.completion_tokens / 1e6 * PRICE_PER_1M_OUTPUT

essay = [{"role": "user", "content": "Write a detailed essay about the history of computing."}]
for limit in [None, 150]:
    r = client.chat.completions.create(model=MODEL, messages=essay, max_tokens=limit)
    print(f"max_tokens={str(limit):5} | output tokens={r.usage.completion_tokens:5} | "
          f"cost=${cost(r.usage):.6f} | per 10,000 calls=${cost(r.usage) * 10_000:.2f} | finish={r.choices[0].finish_reason}")

## What could go wrong? Trusting a cut-off answer
A reply cut off by `max_tokens` looks completely normal if you only read `.content` - no error, no crash. Always check `finish_reason`.

Here's a small helper we'll keep using: it reads the text **and** warns when the answer is incomplete.

In [0]:
def get_reply(response):
    """Return the reply text, warning if the model was cut off."""
    choice = response.choices[0]
    if choice.finish_reason == "length":
        print("WARNING: the answer was cut off by max_tokens - do not trust it as complete.")
    return choice.message.content

short = client.chat.completions.create(model=MODEL, messages=request, max_tokens=40)
print(get_reply(short))

# In Video 08 our agent loop makes decisions based on finish_reason.
# An agent that ignores it acts on half answers.

## Recap
* `choices` is a **list** of answers - usually one, at `[0]`.
* The text is `choices[0].message.content`; `tool_calls` comes in Video 06.
* **`finish_reason`**: `stop` = done, `length` = cut off, `tool_calls` = wants a tool.
* **`usage`** counts tokens (about 3/4 of a word each) - what you pay for.
* **`max_tokens`** caps length and cost; **`temperature`** sets predictable vs creative.

## Practise
* [exercises.md](../exercises.md) - 3 exercises + bonus (about 15 minutes). Add your code below this cell.
* [quiz.md](../quiz.md) - 5 questions.
* Solutions: `03-reading-the-reply-solutions` (try first!).

**Next:** Video 04 - LLMs have no memory.